# Thematic Investing: A Risk-based Perspective 再現実装

このNotebookは、`docs/notes/review.md` に整理した Phase B の最小再現です。日本株テーマバスケット、構成銘柄トータルリターン、Barra GEMLTL エクスポージャーが `data/raw/` に配置されていれば実データを読みます。未配置の場合は、同じスキーマの合成データを生成して、残差推定、APWC、Mosaic + bootstrap、リスク過小評価、残差リターン持続性を通しで検証します。

実装はこのNotebook内で完結させます。外部 `.py` モジュールは使いません。

## 0. 実装方針と時点整合

- 特徴量は分析日 `t` までの残差リターンだけで計算する。
- 将来残差リターンは `t+1` 以降だけを評価ラベルに使う。
- 月次リバランスでは、月末営業日 `t` で決めたシグナルを翌営業日 `t+1` から適用する。
- 初期実装は OLS、60営業日窓、`z >= 2` を coherent theme 判定のデフォルトにする。
- 実データ未配置時は `USING_SYNTHETIC_DATA = True` と表示し、実データ検証は未実施として扱う。

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

PROJECT_ROOT = Path.cwd()
DATA_RAW = PROJECT_ROOT / "data" / "raw"

@dataclass(frozen=True)
class Config:
    seed: int = 42
    window: int = 60
    future_window: int = 60
    block_size: int = 10
    bootstrap_reps_synthetic: int = 40
    bootstrap_reps_real: int = 200
    min_obs_ratio: float = 0.80
    z_threshold: float = 2.0
    trading_days_per_year: int = 252
    max_analysis_dates_per_theme: int = 3

cfg = Config()
rng = np.random.default_rng(cfg.seed)
print(cfg)

## 1. 入力スキーマ

実データを置く場合は、以下の4テーブルを `data/raw/` に `.parquet` または `.csv` で配置します。どれか一部だけ存在する状態は、混在を避けるためエラーにします。

In [ ]:
REQUIRED_SCHEMAS: dict[str, set[str]] = {
    "theme_constituents": {"as_of_date", "theme_id", "theme_name", "release_date", "ticker", "weight"},
    "stock_total_returns": {"date", "ticker", "total_return"},
    "barra_gemltl_exposures": {"date", "ticker", "factor", "exposure"},
    "universe": {"date", "ticker", "in_universe"},
}

DATE_COLUMNS = {
    "theme_constituents": ["as_of_date", "release_date"],
    "stock_total_returns": ["date"],
    "barra_gemltl_exposures": ["date"],
    "universe": ["date"],
}

NUMERIC_COLUMNS = {
    "theme_constituents": ["weight"],
    "stock_total_returns": ["total_return"],
    "barra_gemltl_exposures": ["exposure"],
    "universe": [],
}


def find_table(stem: str) -> Path | None:
    for suffix in (".parquet", ".csv"):
        candidate = DATA_RAW / f"{stem}{suffix}"
        if candidate.exists():
            return candidate
    return None


def read_table(path: Path) -> pd.DataFrame:
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    if path.suffix == ".csv":
        return pd.read_csv(path)
    raise ValueError(f"Unsupported table format: {path}")


def validate_schema(name: str, df: pd.DataFrame) -> pd.DataFrame:
    missing = REQUIRED_SCHEMAS[name] - set(df.columns)
    if missing:
        raise ValueError(f"{name} missing columns: {sorted(missing)}")

    out = df.copy()
    for col in DATE_COLUMNS[name]:
        out[col] = pd.to_datetime(out[col])
    for col in NUMERIC_COLUMNS[name]:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    if name == "universe":
        out["in_universe"] = out["in_universe"].astype(bool)
    return out


def validate_all_tables(tables: dict[str, pd.DataFrame]) -> dict[str, pd.DataFrame]:
    return {name: validate_schema(name, df) for name, df in tables.items()}

schema_table = pd.DataFrame(
    [(name, sorted(cols)) for name, cols in REQUIRED_SCHEMAS.items()],
    columns=["table", "required_columns"],
)
display(schema_table)

## 2. データロードまたは合成データ生成

合成データでは、テーマ `T001` に一時的な共通残差ショックを入れ、coherent theme が検出できるようにします。これは実装検証用であり、研究結果として解釈しません。

In [ ]:
def generate_synthetic_tables(cfg: Config) -> dict[str, pd.DataFrame]:
    local_rng = np.random.default_rng(cfg.seed)
    dates = pd.bdate_range("2022-01-04", periods=360)
    tickers = [f"JP{i:04d}" for i in range(1, 121)]
    factors = ["market", "value", "size", "momentum", "quality"]

    static_exposure = pd.DataFrame(
        local_rng.normal(0.0, 1.0, size=(len(tickers), len(factors))),
        index=tickers,
        columns=factors,
    )
    static_exposure["market"] = 1.0

    exposure_rows = []
    for i, date in enumerate(dates):
        drift = local_rng.normal(0.0, 0.015, size=static_exposure.shape)
        exp = static_exposure + drift
        for ticker in tickers:
            for factor in factors:
                exposure_rows.append((date, ticker, factor, exp.loc[ticker, factor]))
    barra_gemltl_exposures = pd.DataFrame(
        exposure_rows, columns=["date", "ticker", "factor", "exposure"]
    )

    factor_returns = pd.DataFrame(
        local_rng.normal(0.0, 0.003, size=(len(dates), len(factors))),
        index=dates,
        columns=factors,
    )
    factor_returns["market"] = local_rng.normal(0.0002, 0.006, size=len(dates))

    theme_specs = {
        "T001": {
            "theme_name": "Synthetic Coherent AI Infrastructure",
            "members": tickers[:12],
            "release_idx": 80,
            "common_scale": 0.018,
        },
        "T002": {
            "theme_name": "Synthetic Moderate Green Transition",
            "members": tickers[12:26],
            "release_idx": 95,
            "common_scale": 0.006,
        },
        "T003": {
            "theme_name": "Synthetic Non Coherent Domestic Demand",
            "members": tickers[80:92],
            "release_idx": 190,
            "common_scale": 0.000,
        },
    }

    common_shocks: dict[str, np.ndarray] = {}
    event_start, event_end = 75, 180
    for theme_id, spec in theme_specs.items():
        shock = np.zeros(len(dates))
        for t in range(1, len(dates)):
            innovation_scale = spec["common_scale"] if event_start <= t <= event_end else spec["common_scale"] * 0.15
            shock[t] = 0.55 * shock[t - 1] + local_rng.normal(0.0, innovation_scale)
        common_shocks[theme_id] = shock

    return_rows = []
    for t, date in enumerate(dates):
        exp_today = barra_gemltl_exposures[barra_gemltl_exposures["date"].eq(date)].pivot(
            index="ticker", columns="factor", values="exposure"
        )[factors]
        systematic = exp_today.to_numpy() @ factor_returns.loc[date, factors].to_numpy()
        residual = local_rng.normal(0.0, 0.008, size=len(tickers))
        for theme_id, spec in theme_specs.items():
            member_idx = [tickers.index(x) for x in spec["members"]]
            residual[member_idx] += common_shocks[theme_id][t]
        total_return = systematic + residual
        return_rows.extend((date, ticker, ret) for ticker, ret in zip(tickers, total_return))
    stock_total_returns = pd.DataFrame(return_rows, columns=["date", "ticker", "total_return"])

    constituent_rows = []
    for theme_id, spec in theme_specs.items():
        release_date = dates[spec["release_idx"]]
        weight = 1.0 / len(spec["members"])
        for ticker in spec["members"]:
            constituent_rows.append(
                (release_date, theme_id, spec["theme_name"], release_date, ticker, weight)
            )
    theme_constituents = pd.DataFrame(
        constituent_rows,
        columns=["as_of_date", "theme_id", "theme_name", "release_date", "ticker", "weight"],
    )

    universe = pd.MultiIndex.from_product([dates, tickers], names=["date", "ticker"]).to_frame(index=False)
    universe["in_universe"] = True

    return {
        "theme_constituents": theme_constituents,
        "stock_total_returns": stock_total_returns,
        "barra_gemltl_exposures": barra_gemltl_exposures,
        "universe": universe,
    }


def load_or_generate_tables(cfg: Config) -> tuple[dict[str, pd.DataFrame], bool, dict[str, Path | None]]:
    paths = {name: find_table(name) for name in REQUIRED_SCHEMAS}
    present = {name: path for name, path in paths.items() if path is not None}
    if len(present) == len(REQUIRED_SCHEMAS):
        tables = {name: read_table(path) for name, path in present.items()}
        return validate_all_tables(tables), False, paths
    if len(present) == 0:
        tables = generate_synthetic_tables(cfg)
        return validate_all_tables(tables), True, paths
    missing = sorted(set(REQUIRED_SCHEMAS) - set(present))
    raise FileNotFoundError(
        "実データを使う場合は4テーブルをすべて配置してください。"
        f" present={sorted(present)}, missing={missing}, data_dir={DATA_RAW}"
    )


tables, USING_SYNTHETIC_DATA, input_paths = load_or_generate_tables(cfg)
print(f"USING_SYNTHETIC_DATA = {USING_SYNTHETIC_DATA}")
print(f"DATA_RAW = {DATA_RAW}")
display(pd.DataFrame({"table": list(input_paths), "path": [str(v) if v else None for v in input_paths.values()]}))
display(pd.DataFrame({name: [df.shape[0], df.shape[1]] for name, df in tables.items()}, index=["rows", "cols"]).T)

In [ ]:
theme_constituents = tables["theme_constituents"]
stock_total_returns = tables["stock_total_returns"]
barra_gemltl_exposures = tables["barra_gemltl_exposures"]
universe = tables["universe"]

schema_checks = []
for name, df in tables.items():
    schema_checks.append(
        {
            "table": name,
            "rows": len(df),
            "unique_dates": df["date"].nunique() if "date" in df.columns else df["as_of_date"].nunique(),
            "unique_tickers": df["ticker"].nunique() if "ticker" in df.columns else None,
            "missing_cells": int(df.isna().sum().sum()),
        }
    )

schema_checks = pd.DataFrame(schema_checks)
display(schema_checks)

assert set(theme_constituents["ticker"]).issubset(set(stock_total_returns["ticker"])), "テーマ構成銘柄がリターンに存在しません。"
assert set(stock_total_returns["ticker"]).intersection(set(barra_gemltl_exposures["ticker"])), "リターンとエクスポージャーに共通銘柄がありません。"

## 3. Barra型クロスセクション回帰と残差リターン

各営業日で、同日時点のエクスポージャーを使ってクロスセクションOLSを実行します。ここでは再現の初期版としてOLSに固定します。WLSやBarra固有の推奨ウェイトは、実データ仕様が確定した後の拡張対象です。

In [ ]:
def build_exposure_lookup(
    exposures: pd.DataFrame,
    universe: pd.DataFrame,
) -> tuple[dict[pd.Timestamp, pd.DataFrame], dict[pd.Timestamp, set[str]], list[str]]:
    factor_names = sorted(exposures["factor"].unique())
    exposure_by_date = {
        date: g.pivot(index="ticker", columns="factor", values="exposure").reindex(columns=factor_names)
        for date, g in exposures.groupby("date", sort=True)
    }
    universe_by_date = {
        date: set(g.loc[g["in_universe"], "ticker"])
        for date, g in universe.groupby("date", sort=True)
    }
    return exposure_by_date, universe_by_date, factor_names


def fit_residual_wide_from_returns_wide(
    returns_wide: pd.DataFrame,
    exposure_by_date: dict[pd.Timestamp, pd.DataFrame],
    universe_by_date: dict[pd.Timestamp, set[str]],
    factor_names: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    residual_wide = pd.DataFrame(index=returns_wide.index, columns=returns_wide.columns, dtype=float)
    beta_rows: list[dict[str, Any]] = []
    diag_rows: list[dict[str, Any]] = []

    for date in returns_wide.index:
        y_all = returns_wide.loc[date].dropna()
        exposures = exposure_by_date.get(date)
        universe_set = universe_by_date.get(date, set(y_all.index))
        if exposures is None:
            continue

        tickers = y_all.index.intersection(exposures.dropna().index)
        tickers = [ticker for ticker in tickers if ticker in universe_set]
        if len(tickers) <= len(factor_names) + 1:
            diag_rows.append({"date": date, "n_obs": len(tickers), "status": "too_few_obs"})
            continue

        y = y_all.loc[tickers].astype(float).to_numpy()
        x_factor = exposures.loc[tickers, factor_names].astype(float).to_numpy()
        x = np.column_stack([np.ones(len(tickers)), x_factor])
        beta, *_ = np.linalg.lstsq(x, y, rcond=None)
        fitted = x @ beta
        residual = y - fitted
        residual_wide.loc[date, tickers] = residual

        row = {"date": date, "intercept": beta[0]}
        row.update({factor: beta[i + 1] for i, factor in enumerate(factor_names)})
        beta_rows.append(row)
        diag_rows.append(
            {
                "date": date,
                "n_obs": len(tickers),
                "n_factors": len(factor_names),
                "residual_mean": float(np.mean(residual)),
                "residual_std": float(np.std(residual, ddof=1)),
                "status": "ok",
            }
        )

    factor_returns = pd.DataFrame(beta_rows).set_index("date").sort_index()
    diagnostics = pd.DataFrame(diag_rows).sort_values("date")
    return residual_wide, factor_returns, diagnostics


returns_wide = stock_total_returns.pivot(index="date", columns="ticker", values="total_return").sort_index()
exposure_by_date, universe_by_date, factor_names = build_exposure_lookup(barra_gemltl_exposures, universe)
residuals_wide, factor_returns, regression_diagnostics = fit_residual_wide_from_returns_wide(
    returns_wide, exposure_by_date, universe_by_date, factor_names
)

ok_diag = regression_diagnostics[regression_diagnostics["status"].eq("ok")]
max_abs_residual_mean = ok_diag["residual_mean"].abs().max()
print(f"factors = {factor_names}")
print(f"regression dates ok = {len(ok_diag)} / {len(regression_diagnostics)}")
print(f"max abs daily residual mean = {max_abs_residual_mean:.3e}")
assert max_abs_residual_mean < 1e-10, "OLS残差の日次平均が十分にゼロではありません。"
display(ok_diag.tail())

## 4. APWCとMosaic + bootstrap

APWCは、テーマ構成銘柄の残差相関行列の上三角平均です。Mosaic + bootstrapは、10営業日ブロックと銘柄パネルを使って残差の同時相関を壊し、擬似リターンを再残差化して帰無分布を作ります。

In [ ]:
def latest_constituents(
    constituents: pd.DataFrame,
    theme_id: str,
    analysis_date: pd.Timestamp,
    public_only: bool = True,
) -> pd.DataFrame:
    df = constituents[constituents["theme_id"].eq(theme_id)].copy()
    df = df[df["as_of_date"].le(analysis_date)]
    if public_only:
        df = df[df["release_date"].le(analysis_date)]
    if df.empty:
        return df
    latest_as_of = df["as_of_date"].max()
    df = df[df["as_of_date"].eq(latest_as_of)].copy()
    weight_sum = df["weight"].sum()
    if not np.isfinite(weight_sum) or abs(weight_sum) < 1e-12:
        df["weight"] = 1.0 / len(df)
    else:
        df["weight"] = df["weight"] / weight_sum
    return df


def average_pairwise_corr(window: pd.DataFrame, min_obs_ratio: float = cfg.min_obs_ratio) -> float:
    if window.empty:
        return np.nan
    min_obs = int(np.ceil(len(window) * min_obs_ratio))
    clean = window.dropna(axis=1, thresh=min_obs)
    if clean.shape[1] < 2 or clean.shape[0] < 3:
        return np.nan
    corr = clean.corr(min_periods=min_obs)
    values = corr.to_numpy(dtype=float)
    upper = values[np.triu_indices_from(values, k=1)]
    upper = upper[np.isfinite(upper)]
    if upper.size == 0:
        return np.nan
    return float(np.mean(upper))


def basket_residual_return(residual_window: pd.DataFrame, constituent_df: pd.DataFrame) -> pd.Series:
    tickers = [ticker for ticker in constituent_df["ticker"] if ticker in residual_window.columns]
    if len(tickers) == 0:
        return pd.Series(index=residual_window.index, dtype=float)
    weights = constituent_df.set_index("ticker").loc[tickers, "weight"].astype(float)
    weights = weights / weights.sum()
    return residual_window[tickers].mul(weights, axis=1).sum(axis=1, min_count=1)


def predict_returns_from_factor_model(
    dates: pd.Index,
    columns: pd.Index,
    factor_returns: pd.DataFrame,
    exposure_by_date: dict[pd.Timestamp, pd.DataFrame],
    factor_names: list[str],
) -> pd.DataFrame:
    predicted = pd.DataFrame(index=dates, columns=columns, dtype=float)
    for date in dates:
        if date not in factor_returns.index or date not in exposure_by_date:
            continue
        beta = factor_returns.loc[date]
        exposures = exposure_by_date[date].reindex(columns=factor_names)
        tickers = columns.intersection(exposures.dropna().index)
        if len(tickers) == 0:
            continue
        predicted.loc[date, tickers] = beta["intercept"] + exposures.loc[tickers, factor_names].to_numpy() @ beta[factor_names].to_numpy()
    return predicted


def contiguous_blocks(n: int, block_size: int) -> list[np.ndarray]:
    return [np.arange(start, min(start + block_size, n)) for start in range(0, n, block_size)]


def assign_ticker_panels(tickers: list[str], factor_count: int, rng: np.random.Generator) -> list[list[str]]:
    min_panel_size = max(2, 5 * factor_count)
    n_panels = max(1, len(tickers) // min_panel_size)
    shuffled = np.array(tickers, dtype=object)
    rng.shuffle(shuffled)
    panels = [list(panel) for panel in np.array_split(shuffled, n_panels) if len(panel) > 0]
    return panels


def block_permute_residuals(
    residual_window: pd.DataFrame,
    panels: list[list[str]],
    block_size: int,
    rng: np.random.Generator,
) -> pd.DataFrame:
    out = pd.DataFrame(index=residual_window.index, columns=residual_window.columns, dtype=float)
    blocks = contiguous_blocks(len(residual_window), block_size)
    for panel in panels:
        panel_tickers = [ticker for ticker in panel if ticker in residual_window.columns]
        for ticker in panel_tickers:
            values = residual_window[ticker].to_numpy(dtype=float)
            order = rng.permutation(len(blocks))
            shuffled = np.concatenate([values[blocks[i]] for i in order])[: len(values)]
            out[ticker] = shuffled
    return out


def mosaic_bootstrap_apwc(
    residuals_wide: pd.DataFrame,
    returns_wide: pd.DataFrame,
    factor_returns: pd.DataFrame,
    exposure_by_date: dict[pd.Timestamp, pd.DataFrame],
    universe_by_date: dict[pd.Timestamp, set[str]],
    factor_names: list[str],
    dates: pd.Index,
    basket_tickers: list[str],
    reps: int,
    cfg: Config,
    seed: int,
) -> dict[str, Any]:
    local_rng = np.random.default_rng(seed)
    dates = pd.Index(pd.to_datetime(dates))
    universe_tickers = list(returns_wide.columns)
    panels = assign_ticker_panels(universe_tickers, len(factor_names), local_rng)
    predicted = predict_returns_from_factor_model(dates, returns_wide.columns, factor_returns, exposure_by_date, factor_names)
    residual_window = residuals_wide.loc[dates, returns_wide.columns]

    boot_values: list[float] = []
    for _ in range(reps):
        shuffled_residual = block_permute_residuals(residual_window, panels, cfg.block_size, local_rng)
        synthetic_returns = predicted + shuffled_residual
        boot_resid, _, _ = fit_residual_wide_from_returns_wide(
            synthetic_returns, exposure_by_date, universe_by_date, factor_names
        )
        boot_apwc = average_pairwise_corr(boot_resid.loc[dates, basket_tickers])
        if np.isfinite(boot_apwc):
            boot_values.append(boot_apwc)

    boot = np.array(boot_values, dtype=float)
    return {
        "boot_values": boot,
        "boot_mean": float(np.mean(boot)) if len(boot) else np.nan,
        "boot_std": float(np.std(boot, ddof=1)) if len(boot) > 1 else np.nan,
        "n_boot": int(len(boot)),
        "n_panels": int(len(panels)),
        "panel_sizes": [len(panel) for panel in panels],
    }

## 5. テーマ別のAPWC z-score

分析日は、各テーマの公開日以降で、過去60営業日と将来60営業日が両方取れる日から選びます。これにより、特徴量とラベルの時間方向を明確に分けます。

In [ ]:
def choose_analysis_dates(
    constituents: pd.DataFrame,
    trading_dates: pd.Index,
    cfg: Config,
) -> pd.DataFrame:
    rows = []
    trading_dates = pd.Index(pd.to_datetime(trading_dates)).sort_values()
    for theme_id, g in constituents.groupby("theme_id", sort=True):
        release_date = pd.to_datetime(g["release_date"].min())
        release_pos = int(np.searchsorted(trading_dates.values, np.datetime64(release_date), side="left"))
        first_pos = release_pos + cfg.window
        last_pos = len(trading_dates) - cfg.future_window - 1
        if first_pos > last_pos:
            continue
        candidates = trading_dates[first_pos : last_pos + 1]
        if len(candidates) == 0:
            continue
        if len(candidates) <= cfg.max_analysis_dates_per_theme:
            selected = candidates
        else:
            selected_idx = np.linspace(0, len(candidates) - 1, cfg.max_analysis_dates_per_theme).round().astype(int)
            selected = candidates[selected_idx]
        for analysis_date in selected:
            rows.append(
                {
                    "theme_id": theme_id,
                    "theme_name": g["theme_name"].iloc[0],
                    "release_date": release_date,
                    "analysis_date": pd.Timestamp(analysis_date),
                }
            )
    return pd.DataFrame(rows)


def analyze_theme_date(row: pd.Series, reps: int, cfg: Config) -> dict[str, Any]:
    theme_id = row["theme_id"]
    analysis_date = pd.Timestamp(row["analysis_date"])
    all_dates = residuals_wide.index
    pos = all_dates.get_loc(analysis_date)
    past_dates = all_dates[pos - cfg.window + 1 : pos + 1]
    future_dates = all_dates[pos + 1 : pos + 1 + cfg.future_window]
    assert len(past_dates) == cfg.window
    assert len(future_dates) == cfg.future_window
    assert max(past_dates) <= analysis_date
    assert min(future_dates) > analysis_date

    constituents_now = latest_constituents(theme_constituents, theme_id, analysis_date, public_only=True)
    basket_tickers = [ticker for ticker in constituents_now["ticker"] if ticker in residuals_wide.columns]
    if len(basket_tickers) < 2:
        raise ValueError(f"{theme_id} has fewer than two usable tickers at {analysis_date.date()}")

    observed_apwc = average_pairwise_corr(residuals_wide.loc[past_dates, basket_tickers])
    boot = mosaic_bootstrap_apwc(
        residuals_wide=residuals_wide,
        returns_wide=returns_wide,
        factor_returns=factor_returns,
        exposure_by_date=exposure_by_date,
        universe_by_date=universe_by_date,
        factor_names=factor_names,
        dates=past_dates,
        basket_tickers=basket_tickers,
        reps=reps,
        cfg=cfg,
        seed=cfg.seed + int(pos) + sum(ord(ch) for ch in str(theme_id)),
    )
    z_score = (observed_apwc - boot["boot_mean"]) / boot["boot_std"] if boot["boot_std"] and np.isfinite(boot["boot_std"]) else np.nan

    past_basket = basket_residual_return(residuals_wide.loc[past_dates], constituents_now)
    future_basket = basket_residual_return(residuals_wide.loc[future_dates], constituents_now)

    return {
        "theme_id": theme_id,
        "theme_name": row["theme_name"],
        "release_date": row["release_date"],
        "analysis_date": analysis_date,
        "past_start": pd.Timestamp(past_dates[0]),
        "past_end": pd.Timestamp(past_dates[-1]),
        "future_start": pd.Timestamp(future_dates[0]),
        "future_end": pd.Timestamp(future_dates[-1]),
        "n_members": len(basket_tickers),
        "apwc": observed_apwc,
        "boot_mean": boot["boot_mean"],
        "boot_std": boot["boot_std"],
        "n_boot": boot["n_boot"],
        "n_panels": boot["n_panels"],
        "z_score": z_score,
        "is_coherent": bool(z_score >= cfg.z_threshold) if np.isfinite(z_score) else False,
        "past_residual_return": float(past_basket.sum()),
        "future_residual_return": float(future_basket.sum()),
    }

analysis_calendar = choose_analysis_dates(theme_constituents, residuals_wide.index, cfg)
reps = cfg.bootstrap_reps_synthetic if USING_SYNTHETIC_DATA else cfg.bootstrap_reps_real
print(f"analysis rows = {len(analysis_calendar)}, bootstrap reps per row = {reps}")
display(analysis_calendar)

analysis_results = pd.DataFrame([analyze_theme_date(row, reps, cfg) for _, row in analysis_calendar.iterrows()])
analysis_results["feature_max_date"] = analysis_results["past_end"]
analysis_results["label_min_date"] = analysis_results["future_start"]
assert (analysis_results["feature_max_date"] <= analysis_results["analysis_date"]).all()
assert (analysis_results["label_min_date"] > analysis_results["analysis_date"]).all()

display(
    analysis_results[
        [
            "theme_id",
            "analysis_date",
            "n_members",
            "apwc",
            "boot_mean",
            "boot_std",
            "z_score",
            "is_coherent",
            "past_residual_return",
            "future_residual_return",
        ]
    ]
)

## 6. リスク過小評価の確認

標準リスクモデルが残差共分散の非対角成分をゼロと置く場合と、過去窓の実績残差共分散を使う場合を比較します。

In [ ]:
def residual_risk_comparison(row: pd.Series, cfg: Config) -> dict[str, Any]:
    constituents_now = latest_constituents(theme_constituents, row["theme_id"], row["analysis_date"], public_only=True)
    tickers = [ticker for ticker in constituents_now["ticker"] if ticker in residuals_wide.columns]
    weights = constituents_now.set_index("ticker").loc[tickers, "weight"].astype(float)
    weights = weights / weights.sum()

    window = residuals_wide.loc[(residuals_wide.index >= row["past_start"]) & (residuals_wide.index <= row["past_end"]), tickers]
    window = window.dropna(axis=1, thresh=int(np.ceil(len(window) * cfg.min_obs_ratio)))
    tickers = list(window.columns)
    weights = weights.loc[tickers]
    weights = weights / weights.sum()

    cov = window.cov() * cfg.trading_days_per_year
    diag_cov = pd.DataFrame(np.diag(np.diag(cov.to_numpy())), index=cov.index, columns=cov.columns)
    w = weights.to_numpy()
    model_risk = float(np.sqrt(w @ diag_cov.to_numpy() @ w))
    empirical_risk = float(np.sqrt(w @ cov.to_numpy() @ w))
    return {
        "theme_id": row["theme_id"],
        "analysis_date": row["analysis_date"],
        "is_coherent": row["is_coherent"],
        "apwc": row["apwc"],
        "model_residual_risk": model_risk,
        "empirical_residual_risk": empirical_risk,
        "risk_ratio_empirical_to_model": empirical_risk / model_risk if model_risk > 0 else np.nan,
    }

risk_table = pd.DataFrame([residual_risk_comparison(row, cfg) for _, row in analysis_results.iterrows()])
display(risk_table)

## 7. 残差リターン持続性

論文の Exhibit 9/10 に対応し、`z >= 2` の群とそれ以外の群で、将来60営業日残差リターンを過去60営業日残差リターンで回帰します。

In [ ]:
def persistence_regression(results: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for coherent_flag, g in results.groupby("is_coherent", sort=True):
        g = g.dropna(subset=["past_residual_return", "future_residual_return"])
        if len(g) >= 3 and g["past_residual_return"].std(ddof=1) > 0:
            x = sm.add_constant(g["past_residual_return"])
            y = g["future_residual_return"]
            model = sm.OLS(y, x).fit()
            rows.append(
                {
                    "group": "z >= 2" if coherent_flag else "z < 2",
                    "estimated_coefficient": model.params["past_residual_return"],
                    "t_statistic": model.tvalues["past_residual_return"],
                    "observations": int(model.nobs),
                    "r_squared": model.rsquared,
                }
            )
        else:
            rows.append(
                {
                    "group": "z >= 2" if coherent_flag else "z < 2",
                    "estimated_coefficient": np.nan,
                    "t_statistic": np.nan,
                    "observations": len(g),
                    "r_squared": np.nan,
                }
            )
    return pd.DataFrame(rows)

persistence_table = persistence_regression(analysis_results)
display(persistence_table)

## 8. Coherence条件付きモメンタム診断

APWC上位50%に絞ったモメンタム改善が、既存ファクターではなく残差共通ショックに由来するかを確認する追加診断。既存の `analysis_results`、`residuals_wide`、`returns_wide`、`factor_returns` を再利用し、既存のAPWC/bootstrap実装は変更しない。

合成データ時の数値は研究結果ではなく smoke test として扱う。

In [ ]:
def basket_weighted_return(return_window: pd.DataFrame, constituent_df: pd.DataFrame) -> pd.Series:
    tickers = [ticker for ticker in constituent_df["ticker"] if ticker in return_window.columns]
    if len(tickers) == 0:
        return pd.Series(index=return_window.index, dtype=float)
    weights = constituent_df.set_index("ticker").loc[tickers, "weight"].astype(float)
    weights = weights / weights.sum()
    return return_window[tickers].mul(weights, axis=1).sum(axis=1, min_count=1)


def trading_dates_between(index: pd.Index, start: pd.Timestamp, end: pd.Timestamp) -> pd.Index:
    idx = pd.Index(pd.to_datetime(index)).sort_values()
    return idx[(idx >= pd.Timestamp(start)) & (idx <= pd.Timestamp(end))]


def _standardize_within_date(values: pd.Series) -> pd.Series:
    std = values.std(ddof=0)
    if not np.isfinite(std) or std == 0:
        return pd.Series(0.0, index=values.index)
    return (values - values.mean()) / std


def assign_top50_by_date(df: pd.DataFrame, score_col: str) -> pd.Series:
    flags = pd.Series(False, index=df.index)
    for _, g in df.groupby("analysis_date", sort=True):
        n_top = max(1, int(np.ceil(len(g) * 0.50)))
        top_idx = g[score_col].sort_values(ascending=False, na_position="last").head(n_top).index
        flags.loc[top_idx] = True
    return flags


def build_coherence_momentum_panel(analysis_results: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in analysis_results.iterrows():
        analysis_date = pd.Timestamp(row["analysis_date"])
        constituents_now = latest_constituents(theme_constituents, row["theme_id"], analysis_date, public_only=True)
        past_dates = trading_dates_between(returns_wide.index, row["past_start"], row["past_end"])
        future_dates = trading_dates_between(returns_wide.index, row["future_start"], row["future_end"])
        assert len(past_dates) > 0
        assert len(future_dates) > 0
        assert pd.Timestamp(past_dates.max()) <= analysis_date
        assert pd.Timestamp(future_dates.min()) > analysis_date
        past_total = basket_weighted_return(returns_wide.loc[past_dates], constituents_now)
        future_total = basket_weighted_return(returns_wide.loc[future_dates], constituents_now)
        past_resid = basket_residual_return(residuals_wide.loc[past_dates], constituents_now)
        future_resid = basket_residual_return(residuals_wide.loc[future_dates], constituents_now)
        rows.append(
            {
                "mode": "coherence_momentum_diagnostic",
                "theme_id": row["theme_id"],
                "theme_name": row["theme_name"],
                "analysis_date": analysis_date,
                "past_start": pd.Timestamp(past_dates.min()),
                "past_end": pd.Timestamp(past_dates.max()),
                "future_start": pd.Timestamp(future_dates.min()),
                "future_end": pd.Timestamp(future_dates.max()),
                "apwc": row["apwc"],
                "z_score": row["z_score"],
                "past_total_momentum": float(past_total.sum()),
                "past_residual_momentum": float(past_resid.sum()),
                "future_total_return": float(future_total.sum()),
                "future_residual_return": float(future_resid.sum()),
                "n_members": row["n_members"],
            }
        )
    panel = pd.DataFrame(rows)
    fallback = panel.groupby("analysis_date")["apwc"].transform(_standardize_within_date)
    panel["coherence_z"] = panel["z_score"].where(panel["z_score"].notna(), fallback)
    panel["coherence_top50"] = assign_top50_by_date(panel, "coherence_z")
    assert panel.groupby("analysis_date")["coherence_top50"].sum().min() >= 1
    assert (panel["past_end"] <= panel["analysis_date"]).all()
    assert (panel["future_start"] > panel["analysis_date"]).all()
    return panel


def simple_t_stat(values: pd.Series) -> float:
    values = pd.Series(values).dropna()
    if len(values) < 2:
        return np.nan
    sd = values.std(ddof=1)
    if not np.isfinite(sd) or sd == 0:
        return np.nan
    return float(values.mean() / (sd / np.sqrt(len(values))))


def select_top_momentum_by_date(df: pd.DataFrame, signal_col: str) -> pd.DataFrame:
    selected = []
    for _, g in df.groupby("analysis_date", sort=True):
        if g.empty:
            continue
        n_pick = max(1, int(np.ceil(len(g) * 0.50)))
        selected.append(g.sort_values(signal_col, ascending=False).head(n_pick))
    if not selected:
        return pd.DataFrame(columns=df.columns)
    return pd.concat(selected, ignore_index=True)


def summarize_selected_momentum(selected: pd.DataFrame, label: str) -> dict[str, float | str | int]:
    return {
        "strategy": label,
        "n_obs": int(len(selected)),
        "future_total_return_mean": float(selected["future_total_return"].mean()) if len(selected) else np.nan,
        "future_total_return_t": simple_t_stat(selected["future_total_return"]),
        "future_total_win_rate": float((selected["future_total_return"] > 0).mean()) if len(selected) else np.nan,
        "future_residual_return_mean": float(selected["future_residual_return"].mean()) if len(selected) else np.nan,
        "future_residual_return_t": simple_t_stat(selected["future_residual_return"]),
    }


def build_strategy_return_panel(panel: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for label, base in [("all_momentum", panel), ("coherence_top50_momentum", panel[panel["coherence_top50"]])]:
        for analysis_date, g in base.groupby("analysis_date", sort=True):
            selected = select_top_momentum_by_date(g, "past_total_momentum")
            if selected.empty:
                continue
            factor_window = factor_returns.loc[(factor_returns.index >= selected["future_start"].min()) & (factor_returns.index <= selected["future_end"].max())]
            row = {
                "strategy": label,
                "analysis_date": analysis_date,
                "future_start": selected["future_start"].min(),
                "future_end": selected["future_end"].max(),
                "strategy_return": float(selected["future_total_return"].mean()),
                "strategy_residual_return": float(selected["future_residual_return"].mean()),
                "n_selected": int(len(selected)),
            }
            for col in factor_returns.columns:
                row[f"factor_{col}"] = float(factor_window[col].sum()) if col in factor_window else np.nan
            rows.append(row)
    return pd.DataFrame(rows)


def compare_momentum_by_coherence(panel: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    all_selected = select_top_momentum_by_date(panel, "past_total_momentum")
    top50_selected = select_top_momentum_by_date(panel[panel["coherence_top50"]], "past_total_momentum")
    summary = pd.DataFrame(
        [
            summarize_selected_momentum(all_selected, "all_momentum"),
            summarize_selected_momentum(top50_selected, "coherence_top50_momentum"),
        ]
    )
    strategy_panel = build_strategy_return_panel(panel)
    return summary, strategy_panel


def run_factor_decomposition(strategy_panel: pd.DataFrame) -> pd.DataFrame:
    factor_cols = [c for c in strategy_panel.columns if c.startswith("factor_")]
    rows = []
    for strategy, g in strategy_panel.groupby("strategy", sort=True):
        g = g.dropna(subset=["strategy_return"] + factor_cols)
        if len(g) <= 2 or len(factor_cols) == 0:
            rows.append({"strategy": strategy, "alpha": np.nan, "alpha_t": np.nan, "factor_beta_norm": np.nan, "r_squared": np.nan, "n_obs": int(len(g)), "status": "too_few_obs"})
            continue
        x_raw = g[factor_cols].astype(float)
        keep_cols = [c for c in factor_cols if x_raw[c].std(ddof=0) > 0]
        x_raw = x_raw[keep_cols]
        if x_raw.empty or len(g) <= x_raw.shape[1] + 1:
            rows.append({"strategy": strategy, "alpha": np.nan, "alpha_t": np.nan, "factor_beta_norm": np.nan, "r_squared": np.nan, "n_obs": int(len(g)), "status": "insufficient_df"})
            continue
        x = sm.add_constant(x_raw, has_constant="add")
        model = sm.OLS(g["strategy_return"], x).fit()
        betas = model.params.drop("const", errors="ignore")
        rows.append(
            {
                "strategy": strategy,
                "alpha": float(model.params.get("const", np.nan)),
                "alpha_t": float(model.tvalues.get("const", np.nan)),
                "factor_beta_norm": float(np.sqrt(np.sum(np.square(betas)))) if len(betas) else np.nan,
                "r_squared": float(model.rsquared),
                "n_obs": int(model.nobs),
                "status": "ok",
            }
        )
    return pd.DataFrame(rows)


def run_interaction_regression(panel: pd.DataFrame) -> pd.DataFrame:
    design = panel[["future_residual_return", "past_residual_momentum", "coherence_z"]].copy().dropna()
    design["interaction"] = design["past_residual_momentum"] * design["coherence_z"]
    required = {"future_residual_return", "past_residual_momentum", "coherence_z", "interaction"}
    assert required.issubset(design.columns)
    if len(design) <= 4:
        return pd.DataFrame([{"term": "interaction", "coef": np.nan, "t_stat": np.nan, "n_obs": int(len(design)), "status": "too_few_obs"}])
    x = sm.add_constant(design[["past_residual_momentum", "coherence_z", "interaction"]], has_constant="add")
    if np.linalg.matrix_rank(x.to_numpy()) < x.shape[1]:
        return pd.DataFrame([{"term": "interaction", "coef": np.nan, "t_stat": np.nan, "n_obs": int(len(design)), "status": "rank_deficient"}])
    model = sm.OLS(design["future_residual_return"], x).fit()
    return pd.DataFrame(
        [
            {"term": term, "coef": float(model.params[term]), "t_stat": float(model.tvalues[term]), "n_obs": int(model.nobs), "status": "ok"}
            for term in ["past_residual_momentum", "coherence_z", "interaction"]
        ]
    )


def pca_stats_for_window(window: pd.DataFrame) -> dict[str, float]:
    clean = window.dropna(axis=1, thresh=max(3, int(np.ceil(len(window) * cfg.min_obs_ratio))))
    if clean.shape[1] < 2:
        return {"first_eigenvalue": np.nan, "first_pc_share": np.nan, "apwc": np.nan, "n_tickers": clean.shape[1]}
    corr = clean.corr().fillna(0.0)
    eigvals = np.linalg.eigvalsh(corr.to_numpy(dtype=float))[::-1]
    eigvals = np.maximum(eigvals, 0.0)
    total = eigvals.sum()
    return {
        "first_eigenvalue": float(eigvals[0]),
        "first_pc_share": float(eigvals[0] / total) if total > 0 else np.nan,
        "apwc": average_pairwise_corr(clean),
        "n_tickers": int(clean.shape[1]),
    }


def run_residual_pca_diagnostics(panel: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    for _, row in panel.iterrows():
        constituents_now = latest_constituents(theme_constituents, row["theme_id"], row["analysis_date"], public_only=True)
        tickers = [t for t in constituents_now["ticker"] if t in residuals_wide.columns]
        window = residuals_wide.loc[(residuals_wide.index >= row["past_start"]) & (residuals_wide.index <= row["past_end"]), tickers]
        stats = pca_stats_for_window(window)
        stats.update({"theme_id": row["theme_id"], "analysis_date": row["analysis_date"], "coherence_top50": bool(row["coherence_top50"])})
        rows.append(stats)
    detail = pd.DataFrame(rows)
    summary = detail.groupby("coherence_top50").agg(
        rows=("theme_id", "size"),
        mean_first_eigenvalue=("first_eigenvalue", "mean"),
        mean_first_pc_share=("first_pc_share", "mean"),
        mean_apwc=("apwc", "mean"),
    ).reset_index()
    return detail, summary


def run_placebo_matched_baskets(panel: pd.DataFrame, n_placebo_per_row: int = 1, seed: int = 31415) -> tuple[pd.DataFrame, pd.DataFrame]:
    local_rng = np.random.default_rng(seed)
    all_theme_tickers = set(theme_constituents["ticker"])
    eligible = np.array(sorted(set(returns_wide.columns) - all_theme_tickers), dtype=object)
    if len(eligible) == 0:
        eligible = np.array(sorted(returns_wide.columns), dtype=object)
    rows = []
    for _, row in panel.iterrows():
        n = min(int(row["n_members"]), len(eligible))
        for k in range(n_placebo_per_row):
            sample = list(local_rng.choice(eligible, size=n, replace=False))
            past_dates = trading_dates_between(residuals_wide.index, row["past_start"], row["past_end"])
            future_dates = trading_dates_between(residuals_wide.index, row["future_start"], row["future_end"])
            placebo_past = residuals_wide.loc[past_dates, sample].mean(axis=1).sum()
            placebo_future = residuals_wide.loc[future_dates, sample].mean(axis=1).sum()
            placebo_apwc = average_pairwise_corr(residuals_wide.loc[past_dates, sample])
            rows.append(
                {
                    "theme_id": row["theme_id"],
                    "analysis_date": row["analysis_date"],
                    "placebo_id": k,
                    "n_tickers": n,
                    "placebo_apwc": placebo_apwc,
                    "placebo_past_residual_momentum": float(placebo_past),
                    "placebo_future_residual_return": float(placebo_future),
                    "actual_apwc": row["apwc"],
                    "actual_future_residual_return": row["future_residual_return"],
                }
            )
    detail = pd.DataFrame(rows)
    summary = pd.DataFrame(
        [
            {
                "placebo_rows": int(len(detail)),
                "placebo_mean_apwc": float(detail["placebo_apwc"].mean()) if len(detail) else np.nan,
                "actual_mean_apwc": float(panel["apwc"].mean()) if len(panel) else np.nan,
                "placebo_mean_future_residual_return": float(detail["placebo_future_residual_return"].mean()) if len(detail) else np.nan,
                "actual_mean_future_residual_return": float(panel["future_residual_return"].mean()) if len(panel) else np.nan,
            }
        ]
    )
    return detail, summary

coherence_momentum_panel = build_coherence_momentum_panel(analysis_results)
coherence_momentum_panel_available = len(coherence_momentum_panel) > 0
momentum_comparison, strategy_return_panel = compare_momentum_by_coherence(coherence_momentum_panel)
factor_decomposition_table = run_factor_decomposition(strategy_return_panel)
interaction_regression_table = run_interaction_regression(coherence_momentum_panel)
pca_detail, pca_group_summary = run_residual_pca_diagnostics(coherence_momentum_panel)
placebo_detail, placebo_summary = run_placebo_matched_baskets(coherence_momentum_panel, n_placebo_per_row=1, seed=cfg.seed + 555)

coherence_top50_momentum_mean_return = float(momentum_comparison.loc[momentum_comparison["strategy"].eq("coherence_top50_momentum"), "future_total_return_mean"].iloc[0])
all_momentum_mean_return = float(momentum_comparison.loc[momentum_comparison["strategy"].eq("all_momentum"), "future_total_return_mean"].iloc[0])
interaction_term_t_stat = float(interaction_regression_table.loc[interaction_regression_table["term"].eq("interaction"), "t_stat"].iloc[0])
coh_alpha = factor_decomposition_table.loc[factor_decomposition_table["strategy"].eq("coherence_top50_momentum"), "alpha_t"]
factor_decomposition_alpha_t = float(coh_alpha.iloc[0]) if len(coh_alpha) else np.nan
pca_high = pca_group_summary.loc[pca_group_summary["coherence_top50"].eq(True), "mean_first_eigenvalue"]
pca_low = pca_group_summary.loc[pca_group_summary["coherence_top50"].eq(False), "mean_first_eigenvalue"]
pca_high_vs_low_first_eigen_ratio = float(pca_high.iloc[0] / pca_low.iloc[0]) if len(pca_high) and len(pca_low) and pca_low.iloc[0] != 0 else np.nan
coherence_momentum_interpretation_ready = not USING_SYNTHETIC_DATA

assert coherence_momentum_panel.groupby("analysis_date")["coherence_top50"].sum().min() >= 1
assert (coherence_momentum_panel["past_end"] <= coherence_momentum_panel["analysis_date"]).all()
assert (coherence_momentum_panel["future_start"] > coherence_momentum_panel["analysis_date"]).all()
assert {"past_residual_momentum", "coherence_z", "future_residual_return"}.issubset(coherence_momentum_panel.columns)

print("coherence_momentum_interpretation_ready =", coherence_momentum_interpretation_ready)
print("synthetic note: values are smoke-test diagnostics, not empirical findings." if USING_SYNTHETIC_DATA else "real-data mode: diagnostics can be interpreted after data QA.")
display(coherence_momentum_panel[["theme_id", "analysis_date", "apwc", "z_score", "coherence_top50", "past_total_momentum", "future_total_return", "future_residual_return"]])
display(momentum_comparison)
display(strategy_return_panel)
display(factor_decomposition_table)
display(interaction_regression_table)
display(pca_group_summary)
display(placebo_summary)

## 8. 月次リバランス時点の整合例

月末営業日 `signal_date` で計算されたシグナルは、翌営業日 `apply_from` から使います。このNotebookの実証テーブルでも、特徴量期間は `analysis_date` まで、評価ラベルは翌営業日から始まることを検査しています。

In [ ]:
def next_trading_day(date: pd.Timestamp, trading_dates: pd.Index) -> pd.Timestamp | pd.NaT:
    pos = int(np.searchsorted(trading_dates.values, np.datetime64(date), side="right"))
    if pos >= len(trading_dates):
        return pd.NaT
    return pd.Timestamp(trading_dates[pos])

month_ends = pd.Series(residuals_wide.index, index=residuals_wide.index).groupby(residuals_wide.index.to_period("M")).max()
rebalance_calendar = pd.DataFrame({"signal_date": month_ends.values})
rebalance_calendar["apply_from"] = rebalance_calendar["signal_date"].map(lambda x: next_trading_day(pd.Timestamp(x), residuals_wide.index))
rebalance_calendar = rebalance_calendar.dropna().tail(6)
display(rebalance_calendar)

assert (analysis_results["past_end"] <= analysis_results["analysis_date"]).all()
assert (analysis_results["future_start"] > analysis_results["analysis_date"]).all()

## 9. 週次APWCローリングZゲート付きモメンタム戦略

週末営業日 `t` で、公開済みテーマ構成銘柄だけを使って過去60営業日のAPWCとモメンタムを計算し、翌営業日 `t+1` から次の週次リバランス日まで保有する。APWCはテーマごとの過去週次APWC履歴からローリングZスコア化し、ゲートを通過したテーマの中から高モメンタムテーマへ投資する。合成データでの数値は実証結果ではなく、時点整合と集計処理のsmoke testである。

In [ ]:
@dataclass(frozen=True)
class WeeklyMomentumConfig:
    signal_window: int = cfg.window
    apwc_z_lookback_weeks: int = 12
    apwc_z_min_periods: int = 4
    apwc_z_threshold: float = 0.0
    top_momentum_n: int = 1
    min_members: int = 2
    annualization_weeks: int = 52


weekly_cfg = WeeklyMomentumConfig()


def weekly_rebalance_dates(trading_index: pd.Index) -> pd.Index:
    idx = pd.Index(pd.to_datetime(trading_index)).sort_values()
    weekly = idx.to_series(index=idx).groupby(idx.to_period("W-FRI")).max()
    return pd.Index(weekly.dropna()).sort_values()


def add_apwc_rolling_z(panel: pd.DataFrame, strategy_cfg: WeeklyMomentumConfig) -> pd.DataFrame:
    panel = panel.sort_values(["theme_id", "analysis_date"]).copy()
    pieces = []
    for _, g in panel.groupby("theme_id", sort=True):
        g = g.copy()
        apwc = g["apwc"].astype(float)
        history = apwc.shift(1)
        mean = history.rolling(strategy_cfg.apwc_z_lookback_weeks, min_periods=strategy_cfg.apwc_z_min_periods).mean()
        std = history.rolling(strategy_cfg.apwc_z_lookback_weeks, min_periods=strategy_cfg.apwc_z_min_periods).std(ddof=1)
        g["apwc_rolling_mean_prior"] = mean
        g["apwc_rolling_std_prior"] = std
        g["apwc_rolling_z"] = (apwc - mean) / std.replace(0.0, np.nan)
        pieces.append(g)
    out = pd.concat(pieces, ignore_index=True)
    out["apwc_gate"] = out["apwc_rolling_z"] >= strategy_cfg.apwc_z_threshold
    return out.sort_values(["analysis_date", "theme_id"]).reset_index(drop=True)


def build_weekly_coherence_momentum_panel(strategy_cfg: WeeklyMomentumConfig) -> pd.DataFrame:
    all_dates = pd.Index(pd.to_datetime(returns_wide.index)).sort_values()
    rebal_dates = weekly_rebalance_dates(all_dates)
    rows = []
    for i, analysis_date in enumerate(rebal_dates[:-1]):
        pos = all_dates.get_loc(analysis_date)
        if pos < strategy_cfg.signal_window - 1:
            continue
        next_rebalance = pd.Timestamp(rebal_dates[i + 1])
        past_dates = all_dates[pos - strategy_cfg.signal_window + 1 : pos + 1]
        future_dates = all_dates[(all_dates > analysis_date) & (all_dates <= next_rebalance)]
        if len(future_dates) == 0:
            continue
        for theme_id, g in theme_constituents.groupby("theme_id", sort=True):
            constituents_now = latest_constituents(theme_constituents, theme_id, analysis_date, public_only=True)
            if constituents_now.empty:
                continue
            tickers = [ticker for ticker in constituents_now["ticker"] if ticker in residuals_wide.columns]
            if len(tickers) < strategy_cfg.min_members:
                continue
            past_total = basket_weighted_return(returns_wide.loc[past_dates], constituents_now)
            future_total = basket_weighted_return(returns_wide.loc[future_dates], constituents_now)
            past_resid = basket_residual_return(residuals_wide.loc[past_dates], constituents_now)
            future_resid = basket_residual_return(residuals_wide.loc[future_dates], constituents_now)
            rows.append(
                {
                    "mode": "weekly_apwc_z_gated_momentum",
                    "theme_id": theme_id,
                    "theme_name": g["theme_name"].iloc[0],
                    "analysis_date": pd.Timestamp(analysis_date),
                    "rebalance_date": pd.Timestamp(analysis_date),
                    "next_rebalance_date": next_rebalance,
                    "past_start": pd.Timestamp(past_dates[0]),
                    "past_end": pd.Timestamp(past_dates[-1]),
                    "future_start": pd.Timestamp(future_dates[0]),
                    "future_end": pd.Timestamp(future_dates[-1]),
                    "n_members": len(tickers),
                    "apwc": average_pairwise_corr(residuals_wide.loc[past_dates, tickers]),
                    "past_total_momentum": float(past_total.sum()),
                    "past_residual_momentum": float(past_resid.sum()),
                    "future_total_return": float(future_total.sum()),
                    "future_residual_return": float(future_resid.sum()),
                }
            )
    panel = pd.DataFrame(rows)
    if panel.empty:
        return panel
    panel = add_apwc_rolling_z(panel, strategy_cfg)
    assert (panel["past_end"] <= panel["analysis_date"]).all()
    assert (panel["future_start"] > panel["analysis_date"]).all()
    return panel


def weekly_factor_sums(start: pd.Timestamp, end: pd.Timestamp) -> dict[str, float]:
    window = factor_returns.loc[(factor_returns.index >= start) & (factor_returns.index <= end)]
    return {f"factor_{col}": float(window[col].sum()) for col in factor_returns.columns}


def select_weekly_theme(g: pd.DataFrame, strategy_cfg: WeeklyMomentumConfig) -> pd.DataFrame:
    clean = g.dropna(subset=["past_total_momentum", "future_total_return", "future_residual_return"])
    if clean.empty:
        return clean
    return clean.sort_values("past_total_momentum", ascending=False).head(strategy_cfg.top_momentum_n)


def build_weekly_strategy_returns(panel: pd.DataFrame, strategy_cfg: WeeklyMomentumConfig) -> tuple[pd.DataFrame, pd.DataFrame]:
    strategy_rows = []
    selection_rows = []
    if panel.empty:
        return pd.DataFrame(), pd.DataFrame()
    for analysis_date, g in panel.groupby("analysis_date", sort=True):
        candidates = g.dropna(subset=["future_total_return", "future_residual_return"])
        variants = [
            ("weekly_momentum_all", candidates),
            ("weekly_apwc_z_gated_momentum", candidates[candidates["apwc_gate"]]),
        ]
        for strategy, base in variants:
            selected = select_weekly_theme(base, strategy_cfg)
            invested = not selected.empty
            future_start = candidates["future_start"].min() if not candidates.empty else pd.NaT
            future_end = candidates["future_end"].max() if not candidates.empty else pd.NaT
            row = {
                "strategy": strategy,
                "analysis_date": pd.Timestamp(analysis_date),
                "future_start": future_start,
                "future_end": future_end,
                "strategy_return": float(selected["future_total_return"].mean()) if invested else 0.0,
                "strategy_residual_return": float(selected["future_residual_return"].mean()) if invested else 0.0,
                "n_selected": int(len(selected)),
                "invested": bool(invested),
            }
            if invested:
                row.update(weekly_factor_sums(selected["future_start"].min(), selected["future_end"].max()))
                tmp = selected.copy()
                tmp["strategy"] = strategy
                tmp["portfolio_weight"] = 1.0 / len(tmp)
                selection_rows.append(tmp)
            else:
                row.update({f"factor_{col}": 0.0 for col in factor_returns.columns})
            strategy_rows.append(row)
    return pd.DataFrame(strategy_rows), pd.concat(selection_rows, ignore_index=True) if selection_rows else pd.DataFrame()


def summarize_weekly_strategy(strategy_panel: pd.DataFrame, strategy_cfg: WeeklyMomentumConfig) -> pd.DataFrame:
    rows = []
    for strategy, g in strategy_panel.groupby("strategy", sort=True):
        returns = g["strategy_return"].astype(float)
        residual_returns = g["strategy_residual_return"].astype(float)
        vol = returns.std(ddof=1)
        rows.append(
            {
                "strategy": strategy,
                "n_weeks": int(len(g)),
                "n_invested_weeks": int(g["invested"].sum()),
                "invested_rate": float(g["invested"].mean()) if len(g) else np.nan,
                "mean_weekly_return": float(returns.mean()) if len(g) else np.nan,
                "weekly_return_t": simple_t_stat(returns),
                "weekly_win_rate": float((returns > 0).mean()) if len(g) else np.nan,
                "mean_weekly_residual_return": float(residual_returns.mean()) if len(g) else np.nan,
                "weekly_residual_t": simple_t_stat(residual_returns),
                "annualized_return_approx": float(returns.mean() * strategy_cfg.annualization_weeks) if len(g) else np.nan,
                "annualized_vol_approx": float(vol * np.sqrt(strategy_cfg.annualization_weeks)) if len(g) > 1 else np.nan,
                "sharpe_approx": float((returns.mean() / vol) * np.sqrt(strategy_cfg.annualization_weeks)) if len(g) > 1 and np.isfinite(vol) and vol > 0 else np.nan,
                "compound_return": float((1.0 + returns).prod() - 1.0) if len(g) else np.nan,
            }
        )
    return pd.DataFrame(rows)


weekly_signal_panel = build_weekly_coherence_momentum_panel(weekly_cfg)
weekly_strategy_panel, weekly_selected_themes = build_weekly_strategy_returns(weekly_signal_panel, weekly_cfg)
weekly_strategy_summary = summarize_weekly_strategy(weekly_strategy_panel, weekly_cfg)
weekly_factor_decomposition_table = run_factor_decomposition(weekly_strategy_panel) if not weekly_strategy_panel.empty else pd.DataFrame()

weekly_strategy_timing_ok = bool(
    not weekly_signal_panel.empty
    and (weekly_signal_panel["past_end"] <= weekly_signal_panel["analysis_date"]).all()
    and (weekly_signal_panel["future_start"] > weekly_signal_panel["analysis_date"]).all()
)
weekly_momentum_strategy_available = bool(not weekly_strategy_panel.empty and weekly_strategy_panel["strategy"].nunique() == 2)
weekly_baseline_mean_return = float(weekly_strategy_summary.loc[weekly_strategy_summary["strategy"].eq("weekly_momentum_all"), "mean_weekly_return"].iloc[0]) if (not weekly_strategy_summary.empty and weekly_strategy_summary["strategy"].eq("weekly_momentum_all").any()) else np.nan
weekly_gated_mean_return = float(weekly_strategy_summary.loc[weekly_strategy_summary["strategy"].eq("weekly_apwc_z_gated_momentum"), "mean_weekly_return"].iloc[0]) if (not weekly_strategy_summary.empty and weekly_strategy_summary["strategy"].eq("weekly_apwc_z_gated_momentum").any()) else np.nan
weekly_gated_invested_rate = float(weekly_strategy_summary.loc[weekly_strategy_summary["strategy"].eq("weekly_apwc_z_gated_momentum"), "invested_rate"].iloc[0]) if (not weekly_strategy_summary.empty and weekly_strategy_summary["strategy"].eq("weekly_apwc_z_gated_momentum").any()) else np.nan
weekly_gated_vs_baseline_mean_diff = weekly_gated_mean_return - weekly_baseline_mean_return if np.isfinite(weekly_gated_mean_return) and np.isfinite(weekly_baseline_mean_return) else np.nan
weekly_gated_factor_alpha_t = float(weekly_factor_decomposition_table.loc[weekly_factor_decomposition_table["strategy"].eq("weekly_apwc_z_gated_momentum"), "alpha_t"].iloc[0]) if (not weekly_factor_decomposition_table.empty and weekly_factor_decomposition_table["strategy"].eq("weekly_apwc_z_gated_momentum").any()) else np.nan
weekly_momentum_interpretation_ready = not USING_SYNTHETIC_DATA

weekly_gate_diagnostics = weekly_signal_panel.groupby("analysis_date").agg(
    n_themes=("theme_id", "nunique"),
    n_gate=("apwc_gate", "sum"),
    max_apwc_rolling_z=("apwc_rolling_z", "max"),
    mean_apwc=("apwc", "mean"),
).reset_index() if not weekly_signal_panel.empty else pd.DataFrame()

assert weekly_strategy_timing_ok
if not weekly_signal_panel.empty:
    assert weekly_signal_panel["apwc_rolling_z"].notna().any(), "APWCローリングZスコアが計算されていません。"
if not weekly_strategy_panel.empty:
    assert set(["weekly_momentum_all", "weekly_apwc_z_gated_momentum"]).issubset(set(weekly_strategy_panel["strategy"])), "週次戦略の比較対象が不足しています。"

print("weekly APWC rolling-Z gate config")
display(pd.Series(weekly_cfg.__dict__))
print("synthetic note: 合成データ時の週次戦略結果は研究結果ではなく実装smoke testです。")
display(weekly_strategy_summary)
print("weekly factor decomposition")
display(weekly_factor_decomposition_table)
print("weekly gate diagnostics tail")
display(weekly_gate_diagnostics.tail())
print("weekly selected themes tail")
display(weekly_selected_themes[["strategy", "analysis_date", "theme_id", "apwc_rolling_z", "apwc_gate", "past_total_momentum", "future_total_return", "future_residual_return"]].tail(12))


## 10. 結果要約と検証状況

このセルはNotebook実行の検査サマリーです。`USING_SYNTHETIC_DATA = True` の場合、実データスキーマに対する本番データ検証は未実施です。

In [ ]:
validation_report = {
    "using_synthetic_data": USING_SYNTHETIC_DATA,
    "real_data_validation": "not_run_synthetic_mode" if USING_SYNTHETIC_DATA else "schema_loaded_and_executed",
    "n_themes": int(theme_constituents["theme_id"].nunique()),
    "n_tickers": int(stock_total_returns["ticker"].nunique()),
    "n_return_dates": int(stock_total_returns["date"].nunique()),
    "n_analysis_rows": int(len(analysis_results)),
    "n_coherent_rows": int(analysis_results["is_coherent"].sum()),
    "max_abs_daily_residual_mean": float(max_abs_residual_mean),
    "feature_label_timing_ok": bool((analysis_results["feature_max_date"] <= analysis_results["analysis_date"]).all() and (analysis_results["label_min_date"] > analysis_results["analysis_date"]).all()),
    "notebook_scope": "single_notebook_no_external_py",
    "weekly_momentum_strategy_available": bool(weekly_momentum_strategy_available),
    "weekly_rebalance_count": int(weekly_strategy_panel["analysis_date"].nunique()) if not weekly_strategy_panel.empty else 0,
    "weekly_apwc_z_threshold": weekly_cfg.apwc_z_threshold,
    "weekly_baseline_mean_return": weekly_baseline_mean_return,
    "weekly_gated_mean_return": weekly_gated_mean_return,
    "weekly_gated_invested_rate": weekly_gated_invested_rate,
    "weekly_gated_vs_baseline_mean_diff": weekly_gated_vs_baseline_mean_diff,
    "weekly_gated_factor_alpha_t": weekly_gated_factor_alpha_t,
    "weekly_strategy_timing_ok": bool(weekly_strategy_timing_ok),
    "weekly_momentum_interpretation_ready": bool(weekly_momentum_interpretation_ready),
    "coherence_momentum_panel_available": bool(coherence_momentum_panel_available),
    "coherence_top50_momentum_mean_return": coherence_top50_momentum_mean_return,
    "all_momentum_mean_return": all_momentum_mean_return,
    "interaction_term_t_stat": interaction_term_t_stat,
    "factor_decomposition_alpha_t": factor_decomposition_alpha_t,
    "pca_high_vs_low_first_eigen_ratio": pca_high_vs_low_first_eigen_ratio,
    "coherence_momentum_interpretation_ready": bool(coherence_momentum_interpretation_ready),
}

display(pd.Series(validation_report, name="validation_report"))

summary_by_theme = analysis_results.groupby("theme_id").agg(
    theme_name=("theme_name", "first"),
    rows=("theme_id", "size"),
    mean_apwc=("apwc", "mean"),
    mean_z=("z_score", "mean"),
    coherent_rate=("is_coherent", "mean"),
    mean_risk_ratio=("theme_id", lambda s: np.nan),
).drop(columns=["mean_risk_ratio"])
summary_by_theme = summary_by_theme.join(
    risk_table.groupby("theme_id")["risk_ratio_empirical_to_model"].mean().rename("mean_risk_ratio")
)
display(summary_by_theme)
print("\ncoherence_momentum_comparison")
display(momentum_comparison)
print("\ncoherence_factor_decomposition")
display(factor_decomposition_table)
print("\ncoherence_interaction_regression")
display(interaction_regression_table)
print("\ncoherence_pca_group_summary")
display(pca_group_summary)
print("\ncoherence_placebo_summary")
display(placebo_summary)
print("\nweekly_apwc_z_gated_momentum_summary")
display(weekly_strategy_summary)
print("\nweekly_apwc_z_gated_factor_decomposition")
display(weekly_factor_decomposition_table)
